In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/DS102-ML/project/notebook')

print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/DS102-ML/project/notebook


In [3]:
import os
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')


PREPROCESS_PATH = "../models/preprocessing/feature_engineer.pkl"
SCALER_PATH = "../models/preprocessing/scaler.pkl"
TEST_PATH = "../data/test.csv"
class ProcessFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, drop_duplicates=True, verbose=True):
        self.drop_duplicates = drop_duplicates
        self.verbose = verbose

    def fit(self, X, y=None):
        df = X.copy()

        df["is_root"] = (df["userId"] == 0).astype(int)
        df["is_child_process"] = (df["parentProcessId"] != 1).astype(int)
        df["return_is_error"] = (df["returnValue"] < 0).astype(int)

        self.mnt_freq_ = df.groupby("mountNamespace").size()
        self.mnt_unique_process_ = df.groupby("mountNamespace")["processId"].nunique()

        self.user_freq_ = df.groupby("userId").size()
        self.user_unique_process_ = df.groupby("userId")["processId"].nunique()
        self.user_root_ratio_ = df.groupby("userId")["is_root"].mean()
        self.user_error_ratio_ = df.groupby("userId")["return_is_error"].mean()

        self.proc_freq_ = df.groupby("processId").size()
        self.proc_unique_parent_ = df.groupby("processId")["parentProcessId"].nunique()
        self.proc_root_ratio_ = df.groupby("processId")["is_root"].mean()
        self.proc_child_ratio_ = df.groupby("processId")["is_child_process"].mean()
        self.proc_error_ratio_ = df.groupby("processId")["return_is_error"].mean()
        self.proc_avg_args_ = df.groupby("processId")["argsNum"].mean()

        self.parent_freq_ = df.groupby("parentProcessId").size()
        self.parent_unique_child_ = df.groupby("parentProcessId")["processId"].nunique()
        self.parent_root_child_ratio_ = df.groupby("parentProcessId")["is_root"].mean()
        self.parent_error_ratio_ = df.groupby("parentProcessId")["return_is_error"].mean()

        self.global_freq_ = 1
        self.global_ratio_ = df["return_is_error"].mean()
        self.global_args_ = df["argsNum"].mean()

        if self.verbose:
            print("✅ Hoàn tất fit đặc trưng trên tập huấn luyện")

        return self

    def transform(self, X):
        df = X.copy()

        if self.verbose:
            print(f"🔄 Bắt đầu transform {len(df):,} dòng dữ liệu")

        if self.drop_duplicates:
            df = df.drop_duplicates().reset_index(drop=True)

        if "threadId" in df.columns:
            df["same_process_threadId"] = (df["processId"] == df["threadId"]).astype(int)
            df.drop(columns="threadId", inplace=True, errors="ignore")

        df["is_root"] = (df["userId"] == 0).astype(int)
        df["is_child_process"] = (df["parentProcessId"] != 1).astype(int)

        df["return_is_error"] = (df["returnValue"] < 0).astype(int)
        df["return_is_zero"] = (df["returnValue"] == 0).astype(int)
        df["return_is_positive"] = (df["returnValue"] > 0).astype(int)

        df["mnt_freq"] = df["mountNamespace"].map(self.mnt_freq_).fillna(self.global_freq_)
        df["mnt_unique_process"] = df["mountNamespace"].map(self.mnt_unique_process_).fillna(self.global_freq_)

        df["user_freq"] = df["userId"].map(self.user_freq_).fillna(self.global_freq_)
        df["user_unique_process"] = df["userId"].map(self.user_unique_process_).fillna(self.global_freq_)
        df["user_root_ratio"] = df["userId"].map(self.user_root_ratio_).fillna(self.global_ratio_)
        df["user_error_ratio"] = df["userId"].map(self.user_error_ratio_).fillna(self.global_ratio_)

        df["proc_freq"] = df["processId"].map(self.proc_freq_).fillna(self.global_freq_)
        df["proc_unique_parent"] = df["processId"].map(self.proc_unique_parent_).fillna(self.global_freq_)
        df["proc_root_ratio"] = df["processId"].map(self.proc_root_ratio_).fillna(self.global_ratio_)
        df["proc_child_ratio"] = df["processId"].map(self.proc_child_ratio_).fillna(self.global_ratio_)
        df["proc_error_ratio"] = df["processId"].map(self.proc_error_ratio_).fillna(self.global_ratio_)
        df["proc_avg_args"] = df["processId"].map(self.proc_avg_args_).fillna(self.global_args_)

        df["parent_freq"] = df["parentProcessId"].map(self.parent_freq_).fillna(self.global_freq_)
        df["parent_unique_child"] = df["parentProcessId"].map(self.parent_unique_child_).fillna(self.global_freq_)
        df["parent_root_child_ratio"] = df["parentProcessId"].map(
            self.parent_root_child_ratio_
        ).fillna(self.global_ratio_)
        df["parent_error_ratio"] = df["parentProcessId"].map(
            self.parent_error_ratio_
        ).fillna(self.global_ratio_)

        if self.verbose:
            print(f"✅ Hoàn tất transform, kích thước cuối: {df.shape}")

        return df

preprocessor = joblib.load(PREPROCESS_PATH)
scaler = joblib.load(SCALER_PATH)


df_test = pd.read_csv(TEST_PATH)

df_test_transformed = preprocessor.transform(df_test)
df_test_scaled = scaler.transform(df_test_transformed)


🔄 Bắt đầu transform 188,967 dòng dữ liệu
✅ Hoàn tất transform, kích thước cuối: (188967, 28)


In [6]:
import json

MODEL_PATH = "../models/tuned/lightgbm_tuned.pkl"
OUTPUT_PATH = "../data/submition.csv"
META_PATH = "../models/metadata_20251223_194345.json"

model = joblib.load(MODEL_PATH)
meta = json.load(open(META_PATH))
t = meta['models']['LightGBM']['best_threshold']

test = df_test_scaled.copy()
y_proba = model.predict_proba(test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)
df_submit = pd.DataFrame({
    "id": range(len(test)),
    "target": y_pred
})
df_submit.to_csv(OUTPUT_PATH, index=False)
print(f"📄 File submit đã được lưu tại: {OUTPUT_PATH}")
(df_submit["target"] == 1).sum()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


📄 File submit đã được lưu tại: ../data/submition.csv


np.int64(177431)